# 10 — Repeat the missing mixture ratios and mixed log-Mel

Seed 42 already has every PA fraction. Seeds 43 and 44 already have the 50/50 inverted-Mel run. This notebook does not train those again.

It trains eight new checkpoints:

- inverted-Mel at 100%, 75%, and 25% PA, seeds 43 and 44
- mixed log-Mel at the corpus-balanced 50/50 recipe, seeds 43 and 44

The 0% PA row stays at seed 42. Each new checkpoint is scored on the official 2017 and PA evaluation lists, then PA minimum t-DCF is computed. A finished checkpoint is not trained again if you restart the notebook.

The seed also redraws the speaker-disjoint validation split. Expect about eight to ten hours. Do not start another GPU job while this is running.


In [1]:
import json
import sys
from pathlib import Path

HERE = Path.cwd()
if not (HERE / "common.py").exists():
    HERE = Path(r"D:\speaker-verification-system\replay-cnn-baseline\experiments\paper_official_eval")
sys.path.insert(0, str(HERE))

from common import asv17_root, ensure_pa_ready, experiments_dir, pa_root, paper_runs
from compute_pa_tdcf import min_tdcf_for_predictions

COMPARE = experiments_dir() / "lfcc_vs_mel_compare"
sys.path.insert(0, str(COMPARE))
from experiment_lib import eval_checkpoint_on_corpora, train_feature

ensure_pa_ready()
RUNS = COMPARE / "runs"
PROTECTED = {
    (RUNS / name).resolve()
    for name in (
        "inverted_mel",
        "inverted_mel_seed43",
        "inverted_mel_seed44",
        "inverted_mel_mix_pa100",
        "inverted_mel_mix_pa075",
        "inverted_mel_mix_pa025",
        "inverted_mel_mix_pa000",
        "mel",
    )
}

JOBS = []
for seed in (43, 44):
    for fraction, label in ((1.0, "100% PA"), (0.75, "75/25"), (0.25, "25/75")):
        tag = f"pa{int(round(fraction * 100)):03d}_seed{seed}"
        JOBS.append({
            "feature": "inverted_mel",
            "seed": seed,
            "pa_fraction": fraction,
            "label": f"{label} seed {seed}",
            "run_dir": RUNS / f"inverted_mel_mix_{tag}",
            "eval_dir": paper_runs() / f"mixed_imel_mix_{tag}_eval",
        })
for seed in (43, 44):
    JOBS.append({
        "feature": "mel",
        "seed": seed,
        "pa_fraction": None,
        "label": f"mixed log-Mel seed {seed}",
        "run_dir": RUNS / f"mel_seed{seed}",
        "eval_dir": paper_runs() / f"mixed_mel_seed{seed}_eval",
    })

rows = []
for job in JOBS:
    out = job["run_dir"]
    if out.resolve() in PROTECTED:
        raise RuntimeError(f"refusing to write into an existing run: {out}")
    ckpt = out / f"best_{job['feature']}_mixed_2017_pa2019.pt"
    if ckpt.is_file():
        print(f"\n===== keep existing {job['label']} -> {ckpt} =====", flush=True)
    else:
        print(f"\n===== train {job['label']} -> {out} =====", flush=True)
        summary = train_feature(
            job["feature"],
            asv17_root=asv17_root(),
            pa_root=pa_root(),
            output_dir=out,
            epochs=15,
            patience=4,
            batch_size=8,
            max_train=0,
            max_val=0,
            seed=job["seed"],
            pa_fraction=job["pa_fraction"],
            refresh_cache=False,
        )
        ckpt = Path(summary["checkpoint"]).resolve()
    if ckpt.resolve().parent in PROTECTED:
        raise RuntimeError(f"refusing to score a protected checkpoint: {ckpt}")

    eval_out = job["eval_dir"]
    metrics_path = eval_out / "metrics.json"
    pa_scores = eval_out / "pa2019_predictions.csv"
    if metrics_path.is_file() and pa_scores.is_file():
        print(f"===== keep existing scores for {job['label']} =====", flush=True)
        metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
    else:
        print(f"===== score {job['label']} on official eval lists =====", flush=True)
        metrics = eval_checkpoint_on_corpora(
            ckpt,
            asv17_root=asv17_root(),
            pa_root=pa_root(),
            asv17_split="eval",
            pa_split="eval",
            batch_size=16,
            output_dir=eval_out,
            refresh_cache=False,
        )
    row = {
        "feature": job["feature"],
        "seed": job["seed"],
        "pa_fraction": job["pa_fraction"],
        "label": job["label"],
        "checkpoint": str(ckpt),
        "eer_2017": metrics["asvspoof2017"]["eer_percent"],
        "n_2017": metrics["asvspoof2017"]["n"],
        "eer_pa": metrics["pa2019"]["eer_percent"],
        "n_pa": metrics["pa2019"]["n"],
        "min_tdcf": min_tdcf_for_predictions(pa_scores),
    }
    rows.append(row)
    print(json.dumps(row, indent=2), flush=True)

table_path = paper_runs() / "repeated_seeds_eval.json"
table_path.write_text(json.dumps(rows, indent=2), encoding="utf-8")
print("\nWrote", table_path)
print(f"{'Run':<28} {'2017 EER':>10} {'PA EER':>10} {'min t-DCF':>10}")
for row in rows:
    print(
        f"{row['label']:<28} {row['eer_2017']:10.2f} {row['eer_pa']:10.2f} {row['min_tdcf']:10.4f}"
    )


PA dev probe PA_D_0000007 decodes, and 40/40 PA eval probes decode. 2017 eval audio is present.
No stale PA readability caches.

===== train 100% PA seed 43 -> D:\speaker-verification-system\replay-cnn-baseline\experiments\lfcc_vs_mel_compare\runs\inverted_mel_mix_pa100_seed43 =====
Train inverted_mel | device=cuda | amp=True


Checking train audio: 0it [00:00, ?it/s]
Checking train audio: 0it [00:00, ?it/s]


Train: {"total": 86400, "asvspoof2017": 0, "pa2019": 86400, "bonafide": 8640, "spoof": 77760, "speakers": 16}
Val:   {"total": 11694, "asvspoof2017": 894, "pa2019": 10800, "bonafide": 1527, "spoof": 10167, "speakers": 6}


inverted_mel epoch 1/15:   0%|          | 0/10800 [00:00<?, ?it/s]

Scoring:   0%|          | 0/1462 [00:00<?, ?it/s]

...

epoch=15 train_loss=0.0209 val_eer=5.22% thr=0.4997
Done mel. Best mixed-val EER=4.85%
===== score mixed log-Mel seed 44 on official eval lists =====


Checking eval audio: 0it [00:00, ?it/s]


Scoring:   0%|          | 0/832 [00:00<?, ?it/s]

Scoring:   0%|          | 0/8421 [00:00<?, ?it/s]

{
  "feature": "mel",
  "seed": 44,
  "pa_fraction": null,
  "label": "mixed log-Mel seed 44",
  "checkpoint": "D:\\speaker-verification-system\\replay-cnn-baseline\\experiments\\lfcc_vs_mel_compare\\runs\\mel_seed44\\best_mel_mixed_2017_pa2019.pt",
  "eer_2017": 30.139081649727096,
  "n_2017": 13306,
  "eer_pa": 5.544454937247917,
  "n_pa": 134730,
  "min_tdcf": 0.13558040860690762
}

Wrote D:\speaker-verification-system\replay-cnn-baseline\experiments\paper_official_eval\runs\repeated_seeds_eval.json
Run                            2017 EER     PA EER  min t-DCF
100% PA seed 43                   48.38       7.52     0.1923
75/25 seed 43                     25.60       8.17     0.2066
25/75 seed 43                     19.14      11.26     0.2771
100% PA seed 44                   39.44       7.31     0.1768
75/25 seed 44                     30.41       7.80     0.1926
25/75 seed 44                     20.01       9.20     0.2199
mixed log-Mel seed 43             22.59       5.41     0.1